In [ ]:
# Mount Google Drive

from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Import libraries and load processed datasets

import os
import pandas as pd
import numpy as np

PROJECT_DIR = "/content/drive/MyDrive/Colab Notebooks/jigsaw-toxic-classifier"
PROCESSED_DIR = os.path.join(PROJECT_DIR, "data", "processed")

LABEL_COLUMNS = [
    "toxic", "severe_toxic", "obscene",
    "threat", "insult", "identity_hate"
]

train_df = pd.read_csv(os.path.join(PROCESSED_DIR, "train_sub.csv"))
val_df = pd.read_csv(os.path.join(PROCESSED_DIR, "val.csv"))

print("Sampled training dataset:", train_df.shape)
print("Validation dataset:", val_df.shape)

In [ ]:
# Separate input text and multi-label targets

X_train = train_df["comment_text"].fillna("")
y_train = train_df[LABEL_COLUMNS]

X_val = val_df["comment_text"].fillna("")
y_val = val_df[LABEL_COLUMNS]

print("Training text:", X_train.shape)
print("Training labels:", y_train.shape)
print("Validation text:", X_val.shape)
print("Validation labels:", y_val.shape)

In [ ]:
# Configure word and character TF-IDF vectorisers

from sklearn.feature_extraction.text import TfidfVectorizer

word_vectorizer = TfidfVectorizer(
    analyzer="word",
    ngram_range=(1, 2),
    max_features=100000,
    sublinear_tf=True
)

char_vectorizer = TfidfVectorizer(
    analyzer="char",
    ngram_range=(2, 5),
    max_features=100000,
    sublinear_tf=True
)

print("Word TF-IDF configuration:", word_vectorizer.get_params())
print("\nCharacter TF-IDF configuration:", char_vectorizer.get_params())

In [ ]:
# Fit TF-IDF vectorisers on training data

from scipy.sparse import hstack

X_train_word = word_vectorizer.fit_transform(X_train)
X_train_char = char_vectorizer.fit_transform(X_train)

X_val_word = word_vectorizer.transform(X_val)
X_val_char = char_vectorizer.transform(X_val)

X_train_tfidf = hstack([X_train_word, X_train_char]).tocsr()
X_val_tfidf = hstack([X_val_word, X_val_char]).tocsr()

print("Training TF-IDF shape:", X_train_tfidf.shape)
print("Validation TF-IDF shape:", X_val_tfidf.shape)

In [ ]:
# Train One-vs-Rest Logistic Regression

from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression
import time

lr_model = OneVsRestClassifier(
    LogisticRegression(
        C=1.0,
        solver="liblinear",
        max_iter=1000,
        random_state=42
    ),
    n_jobs=-1
)

start_time = time.time()

lr_model.fit(X_train_tfidf, y_train)

lr_training_time = time.time() - start_time

print("Logistic Regression training completed.")
print("Training time (seconds):", round(lr_training_time, 2))

In [ ]:
# Generate Logistic Regression validation predictions

from sklearn.metrics import classification_report

lr_predictions = lr_model.predict(X_val_tfidf)

print("Validation prediction shape:", lr_predictions.shape)
print("\nClassification Report:")
print(classification_report(
    y_val,
    lr_predictions,
    target_names=LABEL_COLUMNS,
    zero_division=0
))

In [ ]:
# Calculate probability-based validation metrics

from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score
)

lr_probabilities = lr_model.predict_proba(X_val_tfidf)

lr_macro_ap = average_precision_score(y_val, lr_probabilities, average="macro")
lr_micro_ap = average_precision_score(y_val, lr_probabilities, average="micro")

lr_macro_f1 = f1_score(y_val, lr_predictions, average="macro", zero_division=0)
lr_micro_f1 = f1_score(y_val, lr_predictions, average="micro", zero_division=0)

print("Logistic Regression — Validation Metrics")
print("Macro AP:", round(lr_macro_ap, 4))
print("Micro AP:", round(lr_micro_ap, 4))
print("Macro F1:", round(lr_macro_f1, 4))
print("Micro F1:", round(lr_micro_f1, 4))

In [ ]:
# Save Logistic Regression baseline results

import json

a1_results = {
    "experiment": "A1",
    "model": "One-vs-Rest Logistic Regression",
    "macro_ap": float(lr_macro_ap),
    "micro_ap": float(lr_micro_ap),
    "macro_f1": float(lr_macro_f1),
    "micro_f1": float(lr_micro_f1),
    "training_time_seconds": float(lr_training_time),
    "random_seed": 42
}

results_dir = os.path.join(PROJECT_DIR, "results", "experiment_logs")
os.makedirs(results_dir, exist_ok=True)

with open(os.path.join(results_dir, "A1_logistic_regression.json"), "w") as file:
    json.dump(a1_results, file, indent=4)

print("A1 results saved successfully.")

In [ ]:
# Configure One-vs-Rest Calibrated Linear SVM

from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.multiclass import OneVsRestClassifier

svm_model = OneVsRestClassifier(
    CalibratedClassifierCV(
        estimator=LinearSVC(
            C=1.0,
            random_state=42
        ),
        method="sigmoid",
        cv=3
    ),
    n_jobs=-1
)

print("Calibrated Linear SVM configured successfully.")

In [ ]:
# Train One-vs-Rest Calibrated Linear SVM

import time

start_time = time.time()

svm_model.fit(X_train_tfidf, y_train)

svm_training_time = time.time() - start_time

print("Calibrated Linear SVM training completed.")
print("Training time (seconds):", round(svm_training_time, 2))

In [ ]:
# Generate calibrated SVM validation predictions

svm_predictions = svm_model.predict(X_val_tfidf)
svm_probabilities = svm_model.predict_proba(X_val_tfidf)

print("Validation prediction shape:", svm_predictions.shape)
print("Validation probability shape:", svm_probabilities.shape)

In [ ]:
# Evaluate Calibrated Linear SVM on validation data

from sklearn.metrics import (
    classification_report,
    average_precision_score,
    f1_score
)

print("Classification Report:\n")
print(classification_report(
    y_val,
    svm_predictions,
    target_names=LABEL_COLUMNS,
    zero_division=0
))

svm_macro_ap = average_precision_score(
    y_val, svm_probabilities, average="macro"
)
svm_micro_ap = average_precision_score(
    y_val, svm_probabilities, average="micro"
)

svm_macro_f1 = f1_score(
    y_val, svm_predictions, average="macro", zero_division=0
)
svm_micro_f1 = f1_score(
    y_val, svm_predictions, average="micro", zero_division=0
)

print("\nSVM Validation Metrics")
print("Macro AP:", round(svm_macro_ap, 4))
print("Micro AP:", round(svm_micro_ap, 4))
print("Macro F1:", round(svm_macro_f1, 4))
print("Micro F1:", round(svm_micro_f1, 4))

In [ ]:
# Save Calibrated Linear SVM baseline results

a2_results = {
    "experiment": "A2",
    "model": "One-vs-Rest Calibrated Linear SVM",
    "macro_ap": float(svm_macro_ap),
    "micro_ap": float(svm_micro_ap),
    "macro_f1": float(svm_macro_f1),
    "micro_f1": float(svm_micro_f1),
    "training_time_seconds": float(svm_training_time),
    "random_seed": 42
}

with open(os.path.join(results_dir, "A2_calibrated_svm.json"), "w") as file:
    json.dump(a2_results, file, indent=4)

print("A2 results saved successfully.")

In [ ]:
# Compare classical baseline experiments

baseline_comparison = pd.DataFrame([
    {
        "Experiment": "A1",
        "Model": "Logistic Regression",
        "Macro AP": lr_macro_ap,
        "Micro AP": lr_micro_ap,
        "Macro F1": lr_macro_f1,
        "Micro F1": lr_micro_f1,
        "Training Time (s)": lr_training_time
    },
    {
        "Experiment": "A2",
        "Model": "Calibrated Linear SVM",
        "Macro AP": svm_macro_ap,
        "Micro AP": svm_micro_ap,
        "Macro F1": svm_macro_f1,
        "Micro F1": svm_micro_f1,
        "Training Time (s)": svm_training_time
    }
])

print(baseline_comparison.round(4).to_string(index=False))

In [ ]:
# Save classical baseline comparison

tables_dir = os.path.join(PROJECT_DIR, "results", "tables")
os.makedirs(tables_dir, exist_ok=True)

baseline_comparison.to_csv(
    os.path.join(tables_dir, "T05_classical_baseline_comparison.csv"),
    index=False
)

print("Baseline comparison table saved successfully.")

In [ ]:
# Create directory for classical baseline models

import os

MODEL_DIR = os.path.join(PROJECT_DIR, "models", "saved_checkpoints")
os.makedirs(MODEL_DIR, exist_ok=True)

print("Model directory ready:", MODEL_DIR)

In [ ]:
# Save fitted TF-IDF vectorisers

import joblib

joblib.dump(
    word_vectorizer,
    os.path.join(MODEL_DIR, "word_tfidf_vectorizer.pkl")
)

joblib.dump(
    char_vectorizer,
    os.path.join(MODEL_DIR, "char_tfidf_vectorizer.pkl")
)

print("Both TF-IDF vectorisers saved successfully.")

In [ ]:
# Save the trained Logistic Regression model

joblib.dump(
    lr_model,
    os.path.join(MODEL_DIR, "A1_logistic_regression.pkl")
)

print("Logistic Regression model saved successfully.")

In [ ]:
# Save the trained Calibrated Linear SVM model

joblib.dump(
    svm_model,
    os.path.join(MODEL_DIR, "A2_calibrated_linear_svm.pkl")
)

print("Calibrated Linear SVM model saved successfully.")

In [ ]:
# Verify saved baseline model files

for filename in [
    "word_tfidf_vectorizer.pkl",
    "char_tfidf_vectorizer.pkl",
    "A1_logistic_regression.pkl",
    "A2_calibrated_linear_svm.pkl"
]:
    filepath = os.path.join(MODEL_DIR, filename)
    print(filename, "—", "Available" if os.path.exists(filepath) else "Missing")

In [ ]:
# Plot Macro Average Precision comparison

import matplotlib.pyplot as plt

graphs_dir = os.path.join(PROJECT_DIR, "results", "graphs")
os.makedirs(graphs_dir, exist_ok=True)

models = ["Logistic Regression", "Calibrated SVM"]
macro_ap_scores = [lr_macro_ap, svm_macro_ap]

plt.figure(figsize=(7, 5))
bars = plt.bar(models, macro_ap_scores)

plt.title("Classical Baselines - Macro Average Precision")
plt.ylabel("Macro AP")
plt.ylim(0, 1)

for bar, score in zip(bars, macro_ap_scores):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.015,
        f"{score:.4f}",
        ha="center"
    )

plt.tight_layout()
plt.savefig(os.path.join(graphs_dir, "G10_macro_ap_comparison.png"), dpi=300)
plt.show()

In [ ]:
# Plot Macro F1 comparison

models = ["Logistic Regression", "Calibrated SVM"]
macro_f1_scores = [lr_macro_f1, svm_macro_f1]

plt.figure(figsize=(7, 5))
bars = plt.bar(models, macro_f1_scores)

plt.title("Classical Baselines - Macro F1 Comparison")
plt.ylabel("Macro F1")
plt.ylim(0, 1)

for bar, score in zip(bars, macro_f1_scores):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.015,
        f"{score:.4f}",
        ha="center"
    )

plt.tight_layout()
plt.savefig(
    os.path.join(graphs_dir, "G11_macro_f1_comparison.png"),
    dpi=300
)
plt.show()

In [ ]:
# Compare per-label F1 scores for both baseline models

from sklearn.metrics import f1_score
import numpy as np

lr_label_f1 = f1_score(
    y_val, lr_predictions, average=None, zero_division=0
)

svm_label_f1 = f1_score(
    y_val, svm_predictions, average=None, zero_division=0
)

x = np.arange(len(LABEL_COLUMNS))
width = 0.35

plt.figure(figsize=(10, 6))

plt.bar(x - width/2, lr_label_f1, width, label="Logistic Regression")
plt.bar(x + width/2, svm_label_f1, width, label="Calibrated SVM")

plt.title("Classical Baselines - Per-Label F1 Comparison")
plt.xlabel("Toxicity Labels")
plt.ylabel("F1 Score")
plt.xticks(x, LABEL_COLUMNS, rotation=20)
plt.ylim(0, 1)
plt.legend()

plt.tight_layout()
plt.savefig(
    os.path.join(graphs_dir, "G12_per_label_f1_comparison.png"),
    dpi=300
)
plt.show()

In [ ]:
# Verify all classical baseline graphs

for filename in [
    "G10_macro_ap_comparison.png",
    "G11_macro_f1_comparison.png",
    "G12_per_label_f1_comparison.png"
]:
    filepath = os.path.join(graphs_dir, filename)
    print(filename, "—", "Available" if os.path.exists(filepath) else "Missing")